# bias-correction-divide — worked example 2: Build a bias-correction factor tensor for steps 1..n with torch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bias-correction-divide`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The inflation factor `1/(1 - beta**t)` can be computed for many steps at once. Putting `t = 1..n` in a tensor and exponentiating elementwise gives the whole warmup schedule in one shot. The factor is largest at `t=1` and decays monotonically toward 1, which is what makes Adam's early steps behave well.

## Worked solution

**Goal:** return a length-`n` tensor whose `i`-th entry is `1/(1 - beta**(i+1))`.

**Step 1 — make the 1-based step indices.** `t.arange(1, n+1)` gives `[1,2,...,n]`. We cast to float so that `beta ** steps` does real exponentiation rather than integer power.

**Step 2 — exponentiate elementwise.** `beta ** steps` broadcasts the scalar `beta` against the step tensor, producing `[beta^1, beta^2, ..., beta^n]`. These shrink toward 0 as `t` grows.

**Step 3 — form the reciprocal denominator.** `1.0 / (1 - beta**steps)` is the inflation factor. Because `beta**t` decreases, `1 - beta**t` increases toward 1, so the reciprocal decreases toward 1 — the monotone decay we expect.

**Step 4 — why a tensor.** Doing it vectorized means a single optimizer can correct every parameter group's schedule without a Python loop, and the result is ready to multiply against stacked moment buffers.

In [ ]:
def inflation_factors(beta, n, dtype=t.float64):
    steps = t.arange(1, n + 1, dtype=dtype)
    return 1.0 / (1.0 - beta ** steps)

factors = inflation_factors(0.9, 5)
print("factors:", factors)
print("monotone decreasing:", bool(t.all(factors[1:] < factors[:-1])))
print("approaches 1:", float(factors[-1]))